In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
mainPath = os.path.abspath(os.path.join(os.getcwd(), '..'))
if mainPath not in sys.path:
    sys.path.insert(0, mainPath)

In [2]:
p = {
    # Infiltration parameter
    # Unit: m3/h/m2, following your calc_ach() function
    "infl_rate_m3ph_m2": 0.5,

    # Ventilation parameters
    "fresh_air_lps": 12.0,
    "atrium_ach_day": 2.0,
    "atrium_ach_night": 0.0,
    "atrium_volume": 5739.53,

    # Heating/cooling setpoints
    "t_set_heating": 20.0,
    "t_setback_heating": 16.0,
    "t_weekend_heating": 18.0,
    "t_set_cooling": 26.0,

    # Zone construction/physics
    "thermal_capacitance_per_floor_area": 165000,
    "u_walls": 0.2,
    "u_windows": 1.1,
    "lighting_load": 11.7,
    "ventilation_efficiency": 0.6,
}

In [3]:
def update_ventilation(zone, ach_vent, ach_infl, vent_eff=None):
    """
    Update timestep ventilation conductance for the existing ETH Zone object.
    """

    if vent_eff is None:
        vent_eff = getattr(zone, "ventilation_efficiency", 0.6)

    ach_tot = ach_vent + ach_infl

    if ach_tot > 0:
        b_ek = 1.0 - (ach_vent / ach_tot) * vent_eff
        h_ve_adj = 1200.0 * b_ek * zone.room_vol * (ach_tot / 3600.0)
    else:
        b_ek = 1.0
        h_ve_adj = 0.0

    zone.ach_vent = ach_vent
    zone.ach_infl = ach_infl
    zone.ventilation_efficiency = vent_eff
    zone.h_ve_adj = h_ve_adj

    # diagnostics
    zone.last_ach_vent = ach_vent
    zone.last_ach_infl = ach_infl
    zone.last_ach_tot = ach_tot
    zone.last_b_ek = b_ek
    zone.last_h_ve_adj = h_ve_adj

    return h_ve_adj

# ETH_package
### Parameter definition

In [4]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import json

ROOT = Path.cwd().parent.resolve()
RC_SIMULATOR = ROOT / "RC_BuildingSimulator" / "rc_simulator"

if str(RC_SIMULATOR) not in sys.path:
    sys.path.insert(0, str(RC_SIMULATOR))
    
from building_physics import Zone
import supply_system
import emission_system
from radiation import Location, Window
from _BR2_ import *

ElectricityOut = []
HeatingDemand  = []
HeatingEnergy  = []
CoolingDemand  = []
CoolingEnergy  = []
IndoorAir      = []
OutsideTemp    = []
SolarGains     = []
COP            = []

In [5]:
# WEATHER
epw_path      = "../_data/_epw/ED-TMYx.2023.epw"
EDI           = Location(epwfile_path=epw_path)
year          = 2023
latitude_deg  = 55.95
longitude_deg = -3.37

# GEOMETRY
GEOMETRY = {
    "WINDOW_AREA":  1501.3,
    "WALL_AREA"  :  5168.5,
    "FLOOR_AREA" :  9474.7,
    "VOLUME"     : 37738.8,
    "_alpha"     :     3, # coefficient for 'area of surfaces facing the room'
    "_beta"      :     0.3, # coefficient for 'south windows'
}

# HEATING
t_m_prev = 20

t_set_heating = 21.0          # weekday occupied setpoint
t_weekend_heating = 18.0      # weekend occupied setpoint
t_setback_heating = 12.0      # unoccupied setback
t_set_cooling = 28.0

year = 2023

heating_index = pd.date_range(
    start=f"{year}-01-01 00:00",
    end=f"{year}-12-31 23:00",
    freq="h"
)

weekday_profile = [t_setback_heating] * 8 + [t_set_heating] * 10 + [t_setback_heating] * 6
weekend_profile = [t_setback_heating] * 8 + [t_weekend_heating] * 10 + [t_setback_heating] * 6

heating_schedule = []

for ts in heating_index:
    if ts.weekday() >= 5:   # Saturday=5, Sunday=6
        heating_schedule.extend(weekend_profile)
    else:
        heating_schedule.extend(weekday_profile)


# MV
hr_val           = 0.75
ach_vent, ach_infl    =  calc_ach(
    n_people          = 865,
    fresh_air_lps     = 12.0,
    atrium_ach        = 2.0,
    atrium_volume     = 5739.53,
    infl_rate_m3ph_m2 = 4.0,
    geometry          = GEOMETRY
)
daily_mech       = [ach_vent]*24 
mech_ach         = daily_mech * 365
hour = 0

# IHG
occupancyProfile = pd.read_csv('/Users/rui.bo/Desktop/Learn/RC_BuildingSimulator/rc_simulator/auxiliary/schedules_el_OFFICE.csv')

gain_per_person  = 100
max_occupancy    = 765
appliance_gains  = 11.77


hr_eff           = []
for m_days, eff_winter in [
    (31,hr_val),(28,hr_val),(31,hr_val),
    (30,hr_val),(31,0.0),(30,0.0),
    (31,0.0),(31,0.0),(30,0.0),
    (31,hr_val),(30,hr_val),(31,hr_val)
]:
    for _ in range(m_days * 24):
        if mech_ach[hour] > 0:
            hr_eff.append(eff_winter)
        else:
            hr_eff.append(0.0)
        hour += 1
# NO DIFFERENCE IF PLACING [0]*6 + [ach_vent]*8 + [0]*10 + [ach_vent]*6

In [6]:
Office  = Zone(
    window_area                        = GEOMETRY["WINDOW_AREA"],
    walls_area                         = GEOMETRY["WALL_AREA"],
    floor_area                         = GEOMETRY["FLOOR_AREA"],
    room_vol                           = GEOMETRY["VOLUME"],
    total_internal_area                = GEOMETRY["FLOOR_AREA"]*GEOMETRY["_alpha"],
    thermal_capacitance_per_floor_area = 165000,
    u_walls                            = 0.19,
    u_windows                          = 1.41,
    ach_vent                           = ach_vent,
    ach_infl                           = ach_infl,
    lighting_load                      = 4.25,
    t_set_heating                      = 21.0,
    t_set_cooling                      = 28.0,
    ventilation_efficiency             = 0.5,
    max_cooling_energy_per_floor_area  = -np.inf,
    max_heating_energy_per_floor_area  = np.inf,
    heating_supply_system              = supply_system.DirectHeater,
    cooling_supply_system              = supply_system.DirectCooler,
    heating_emission_system            = emission_system.AirConditioning,
    cooling_emission_system            = emission_system.AirConditioning,
)

SouthWindow  = Window(
    azimuth_tilt                      = 0,
    alititude_tilt                    = 90,
    glass_solar_transmittance         = 0.7,
    glass_light_transmittance         = 0.8,
    area                              = GEOMETRY["WINDOW_AREA"] * GEOMETRY["_beta"],
)

### Simulation

Ventilation is not calculated as an airflow network. It is simplified into a single heat-loss conductance between indoor air and supply/outdoor air

In [7]:
def run_case(
    case_name,
    dynamic_ventilation,
    p,
    GEOMETRY,
    Zone,
    supply_system,
    emission_system,
    EDI,
    SouthWindow,
    occupancyProfile,
    heating_schedule,
    year,
    latitude_deg,
    longitude_deg,
    max_occupancy,
    gain_per_person,
    appliance_gains,
):
    """
    Run one 8760-hour simulation.

    dynamic_ventilation=False:
        original ETH behaviour: fixed ach_vent and ach_infl.

    dynamic_ventilation=True:
        timestep behaviour: ach_vent changes with occupancy and atrium schedule.
    """

    # Fixed/default ventilation for initial zone creation
    ach_vent_fixed, ach_infl_fixed = calc_ach(
        n_people=max_occupancy,
        fresh_air_lps=p["fresh_air_lps"],
        atrium_ach=p["atrium_ach_day"],
        atrium_volume=p["atrium_volume"],
        infl_rate_m3ph_m2=p["infl_rate_m3ph_m2"],
        geometry=GEOMETRY,
    )

    Office = make_zone(
        p=p,
        geometry=GEOMETRY,
        ach_vent=ach_vent_fixed,
        ach_infl=ach_infl_fixed,
        Zone=Zone,
        supply_system=supply_system,
        emission_system=emission_system,
    )

    # Ensure these attributes exist
    Office.ach_vent = ach_vent_fixed
    Office.ach_infl = ach_infl_fixed
    Office.ventilation_efficiency = p["ventilation_efficiency"]

    # Reset states and outputs
    t_m_prev = 20.0

    HeatingDemand = []
    HeatingEnergy = []
    CoolingDemand = []
    CoolingEnergy = []
    ElectricityOut = []
    IndoorAir = []
    OutsideTemp = []
    SolarGains = []
    COP = []

    ACHVent = []
    ACHInfl = []
    HVEAdj = []
    People = []

    for hour in range(8760):
        Office.t_set_heating = heating_schedule[hour]

        occupancy = occupancyProfile.loc[hour, "People"] * max_occupancy

        internal_gains = (
            occupancy * gain_per_person
            + appliance_gains * Office.floor_area
        )

        if dynamic_ventilation:
            # Dynamic ventilation:
            # people-based mechanical ventilation changes with occupancy;
            # atrium ventilation is off when occupancy is zero.
            atrium_ach_t = (
                p["atrium_ach_night"]
                if occupancy <= 0
                else p["atrium_ach_day"]
            )

            ach_vent_t, ach_infl_t = calc_ach(
                n_people=occupancy,
                fresh_air_lps=p["fresh_air_lps"],
                atrium_ach=atrium_ach_t,
                atrium_volume=p["atrium_volume"],
                infl_rate_m3ph_m2=p["infl_rate_m3ph_m2"],
                geometry=GEOMETRY,
            )

            update_ventilation(
                Office,
                ach_vent=ach_vent_t,
                ach_infl=ach_infl_t,
                vent_eff=p["ventilation_efficiency"],
            )

        else:
            # Original fixed ventilation:
            # no timestep update. h_ve_adj remains as initialised in Zone.
            ach_vent_t = ach_vent_fixed
            ach_infl_t = ach_infl_fixed

        t_out = EDI.weather_data["drybulb_C"][hour]

        altitude, azimuth = EDI.calc_sun_position(
            latitude_deg=latitude_deg,
            longitude_deg=longitude_deg,
            year=year,
            hoy=hour,
        )

        SouthWindow.calc_solar_gains(
            sun_altitude=altitude,
            sun_azimuth=azimuth,
            normal_direct_radiation=EDI.weather_data["dirnorrad_Whm2"][hour],
            horizontal_diffuse_radiation=EDI.weather_data["difhorrad_Whm2"][hour],
        )

        SouthWindow.calc_illuminance(
            sun_altitude=altitude,
            sun_azimuth=azimuth,
            normal_direct_illuminance=EDI.weather_data["dirnorillum_lux"][hour],
            horizontal_diffuse_illuminance=EDI.weather_data["difhorillum_lux"][hour],
        )

        Office.solve_energy(
            internal_gains=internal_gains,
            solar_gains=SouthWindow.solar_gains,
            t_out=t_out,
            t_m_prev=t_m_prev,
        )

        Office.solve_lighting(
            illuminance=SouthWindow.transmitted_illuminance,
            occupancy=occupancy,
        )

        t_m_prev = Office.t_m_next

        HeatingDemand_kWh_m2 = (Office.heating_demand / 1000.0) / GEOMETRY["FLOOR_AREA"]
        HeatingEnergy_kWh_m2 = (Office.heating_energy / 1000.0) / GEOMETRY["FLOOR_AREA"]
        CoolingDemand_kWh_m2 = (Office.cooling_demand / 1000.0) / GEOMETRY["FLOOR_AREA"]
        CoolingEnergy_kWh_m2 = (Office.cooling_energy / 1000.0) / GEOMETRY["FLOOR_AREA"]
        ElectricityOut_kWh_m2 = (Office.electricity_out / 1000.0) / GEOMETRY["FLOOR_AREA"]

        HeatingDemand.append(HeatingDemand_kWh_m2)
        HeatingEnergy.append(HeatingEnergy_kWh_m2)
        CoolingDemand.append(CoolingDemand_kWh_m2)
        CoolingEnergy.append(CoolingEnergy_kWh_m2)
        ElectricityOut.append(ElectricityOut_kWh_m2)
        IndoorAir.append(Office.t_air)
        OutsideTemp.append(t_out)
        SolarGains.append(SouthWindow.solar_gains)
        COP.append(Office.cop)

        ACHVent.append(ach_vent_t)
        ACHInfl.append(ach_infl_t)
        People.append(occupancy)
        HVEAdj.append(getattr(Office, "last_h_ve_adj", Office.h_ve_adj))

    results = pd.DataFrame({
        "case": case_name,
        "hour": range(8760),
        "people": People,
        "ach_vent": ACHVent,
        "ach_infl": ACHInfl,
        "h_ve_adj": HVEAdj,
        "t_out": OutsideTemp,
        "t_air": IndoorAir,
        "solar_gains": SolarGains,
        "heating_demand_kWh_m2": HeatingDemand,
        "heating_energy_kWh_m2": HeatingEnergy,
        "cooling_demand_kWh_m2": CoolingDemand,
        "cooling_energy_kWh_m2": CoolingEnergy,
        "electricity_out_kWh_m2": ElectricityOut,
        "cop": COP,
    })

    return results

In [8]:
res_original = run_case(
    case_name="original_fixed_ventilation",
    dynamic_ventilation=False,
    p=p,
    GEOMETRY=GEOMETRY,
    Zone=Zone,
    supply_system=supply_system,
    emission_system=emission_system,
    EDI=EDI,
    SouthWindow=SouthWindow,
    occupancyProfile=occupancyProfile,
    heating_schedule=heating_schedule,
    year=year,
    latitude_deg=latitude_deg,
    longitude_deg=longitude_deg,
    max_occupancy=max_occupancy,
    gain_per_person=gain_per_person,
    appliance_gains=appliance_gains,
)

res_dynamic = run_case(
    case_name="dynamic_ventilation",
    dynamic_ventilation=True,
    p=p,
    GEOMETRY=GEOMETRY,
    Zone=Zone,
    supply_system=supply_system,
    emission_system=emission_system,
    EDI=EDI,
    SouthWindow=SouthWindow,
    occupancyProfile=occupancyProfile,
    heating_schedule=heating_schedule,
    year=year,
    latitude_deg=latitude_deg,
    longitude_deg=longitude_deg,
    max_occupancy=max_occupancy,
    gain_per_person=gain_per_person,
    appliance_gains=appliance_gains,
)

# Output

In [9]:
summary = pd.DataFrame({
    "Original_fixed": {
        "HeatingEnergy": res_original["heating_energy_kWh_m2"].sum(),
        "CoolingEnergy": res_original["cooling_energy_kWh_m2"].sum(),
        "Mean ACH vent": res_original["ach_vent"].mean(),
        "Mean ACH infl": res_original["ach_infl"].mean(),
        "Mean h_ve_adj": res_original["h_ve_adj"].mean(),
        "Mean indoor T": res_original["t_air"].mean(),
    },
    "Dynamic_ventilation": {
        "HeatingEnergy": res_dynamic["heating_energy_kWh_m2"].sum(),
        "CoolingEnergy": res_dynamic["cooling_energy_kWh_m2"].sum(),
        "Mean ACH vent": res_dynamic["ach_vent"].mean(),
        "Mean ACH infl": res_dynamic["ach_infl"].mean(),
        "Mean h_ve_adj": res_dynamic["h_ve_adj"].mean(),
        "Mean indoor T": res_dynamic["t_air"].mean(),
    },
})

summary["Difference"] = summary["Dynamic_ventilation"] - summary["Original_fixed"]
summary

,Original_fixed,Dynamic_ventilation,Difference
HeatingEnergy,1.646077,0.000000,-1.646077
CoolingEnergy,15.616805,48.956364,33.339559
Mean ACH vent,1.179875,0.454073,-0.725802
Mean ACH infl,0.068477,0.068477,0.000000
Mean h_ve_adj,6798.358000,3146.237101,-3652.120899
Mean indoor T,23.456255,25.945670,2.489415
